# B10：Google Workspace AI 整合架構 — Google Doc 對話助理設計

> **場景：** 台灣大型顧問公司（1,000 名顧問），希望在 Google Workspace 內嵌入 AI 助理：  
> (1) Google Docs 內可以對話、分析、修改長篇報告  
> (2) Gmail 自動起草客戶回覆  
> (3) Google Sheets 做資料分析  
> **安全要求：** 客戶資料（NDA 簽的策略報告）不能送到第三方 AI 服務，必須留在 GCP 環境。

## 這本 Notebook 要回答的核心問題

每個架構決策都會問：**「為什麼用 X，不用 Y？」**

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 整合模式 | MCP Server（Google Workspace MCP） | 直接硬編 Workspace REST API |
| 文件 Context 載入 | Progressive Loading（按游標位置動態載入） | 整份文件塞 Context |
| 模型部署 | Vertex AI（GCP, asia-east1） | OpenAI API / Claude API |
| 認證架構 | OAuth 2.0（用戶身份）+ Service Account（後端操作） | 共用 API Key |
| 回應方式 | Streaming Response（token by token） | 等全部生成再顯示 |

In [ ]:
import asyncio
import time
import json
import re
import hashlib
from typing import AsyncGenerator, Optional
from dataclasses import dataclass, field
from enum import Enum

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # 模擬 Vertex AI Gemini Pro
    LLM_AVAILABLE = True
    print("✅ LLM 可用（模擬 Vertex AI Gemini Pro）")
except Exception:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬輸出")

print("\n場景：台灣大型顧問公司 Google Workspace AI 助理")
print("規模：1,000 名顧問 | 安全要求：資料不出 GCP asia-east1")

## 系統架構全貌

```
Google Workspace（Browser Extension / Google Add-on）
    │  OAuth 2.0 token（代表顧問用戶身份）
    ▼
┌─────────────────────────────────────────────────┐
│  FDE Backend（Cloud Run, GCP asia-east1）        │
│                                                  │
│  ┌──────────────┐    ┌───────────────────────┐  │
│  │  MCP Server  │←──→│  Context Manager      │  │
│  │  (Workspace  │    │  Progressive Loader   │  │
│  │   Tools)     │    │  Chunk Selector       │  │
│  └──────┬───────┘    └──────────┬────────────┘  │
│         │                       │               │
│         └───────────┬───────────┘               │
│                     │                           │
│              ┌──────▼──────┐                    │
│              │ Vertex AI   │                    │
│              │ Gemini Pro  │                    │
│              │ (Streaming) │                    │
│              └─────────────┘                    │
└─────────────────────────────────────────────────┘
    │  SSE Streaming
    ▼
Google Docs / Gmail / Sheets（即時顯示）
```

**關鍵設計原則：**
- Cloud Run 在 `asia-east1`（台灣），所有資料不離開 GCP 台灣 Region
- MCP Server 封裝 Google Workspace API，讓 Gemini 只需呼叫工具名稱
- OAuth 2.0 代表用戶身份，確保 AI 只能讀取「該顧問有權限的文件」
- SSE Streaming 確保顧問即時看到回應，不需等待 15 秒

## 決策 1：MCP Server vs. 直接硬編 Workspace REST API

**❌ 被拒絕的方案：直接呼叫 Google Workspace REST API**

在 Agent 的 System Prompt 裡描述每個 API endpoint，讓 LLM 自己組 HTTP request：
- `GET https://docs.googleapis.com/v1/documents/{documentId}` 讀文件
- `PATCH https://docs.googleapis.com/v1/documents/{documentId}:batchUpdate` 修改文件
- `GET https://gmail.googleapis.com/gmail/v1/users/{userId}/messages` 讀 Gmail

**問題：**
- LLM 需要知道所有 API 的細節（endpoint 格式、request body schema、pagination 邏輯）
- Google Workspace API 版本升級（v1 → v2）要改所有 Prompt
- 沒有型別檢查，LLM 產生的 JSON 可能格式錯誤才在執行期發現
- 不同 Workspace 產品（Docs / Gmail / Sheets）API 風格各異，Prompt 越來越長

**✅ 選擇的方案：MCP Server（Model Context Protocol）**

將 Workspace 操作封裝為語意化工具：
- `read_document(doc_id, range?)` → 讀文件段落
- `insert_text(doc_id, text, position)` → 在指定位置插入文字
- `get_email_thread(thread_id)` → 讀郵件串
- `update_sheet_range(sheet_id, range, values)` → 更新試算表

LLM 只需知道工具名稱和語意，API 細節完全由 MCP Server 處理。

In [ ]:
# ─────────────────────────────────────────────────
# MCP Server：封裝 Google Workspace 工具
# 生產環境使用 google-workspace-mcp 套件
# 這裡模擬工具呼叫的行為
# ─────────────────────────────────────────────────

class MockWorkspaceMCPServer:
    """
    模擬 Google Workspace MCP Server。
    生產環境：
      - 部署為 Cloud Run sidecar 或獨立 Cloud Run service
      - 透過 google-auth-library 使用 OAuth token
      - 實作 MCP protocol over HTTP/SSE
    """

    def __init__(self):
        # 模擬文件資料庫
        self._documents = {
            "doc_abc123": {
                "title": "台灣電信業數位轉型策略報告_2025Q4",
                "sections": [
                    {"heading": "1. 執行摘要",      "content": "本報告針對台灣三大電信業者（中華電信、台灣大哥大、遠傳）進行策略分析。整體市場呈現飽和態勢，5G 滲透率達 45%，但 ARPU 下滑 8% YoY。", "char_start": 0,    "char_end": 200},
                    {"heading": "2. 市場分析",      "content": "5G 用戶數已達 1,200 萬，佔智慧型手機用戶 52%。但各業者間的 5G 差異化不明顯，價格競爭持續。B2B 物聯網市場具高成長潛力，預計 2026 年達 120 億台幣。", "char_start": 200,  "char_end": 450},
                    {"heading": "3. 競爭風險分析",  "content": "主要風險：(1) 虛擬電信商 (MVNO) 市場份額從 3% 成長至 8%，低價侵蝕；(2) Google Fi 進入台灣市場的可能性；(3) LINE Mobile 擴大合作範圍；(4) 固網+行動整合包被競爭對手搶先推出。次要風險：監管政策收緊，頻譜費用上調。", "char_start": 450,  "char_end": 750},
                    {"heading": "4. 策略建議",      "content": "建議中華電信聚焦三個方向：(1) 企業 5G 專網（Smart Factory）；(2) 跨境漫遊協議優化；(3) 數位金融服務整合（中華電信 Pay）。預估 3 年 ROI 達 185%。", "char_start": 750,  "char_end": 1000},
                    {"heading": "5. 執行計劃",      "content": "Q1 2026：啟動 5G 專網試點（3 家工廠）。Q2 2026：推出整合資費方案。Q3 2026：數位金融 beta 測試。Q4 2026：全面上線，目標 B2B 營收增加 22%。", "char_start": 1000, "char_end": 1200},
                ],
                "cursor_position": 550,  # 顧問目前游標在第 3 節
            }
        }
        self._emails = {
            "thread_xyz789": [
                {"from": "client@chunghwa.com.tw", "subject": "Re: 策略報告初稿反饋",
                 "body": "您好，感謝提供初稿。第三節的競爭風險分析我們有一些疑問，MVNO 的資料來源是什麼？另外 Google Fi 進台灣的可能性評估依據為何？"},
            ]
        }
        self._sheets = {}

    # ── 工具定義（MCP tool_registry）──────────────────
    @property
    def tool_registry(self) -> dict:
        """MCP 工具清單，供 LLM function calling 使用"""
        return {
            "read_document": {
                "description": "讀取 Google Doc 文件內容。可指定段落範圍。",
                "parameters": {
                    "doc_id": "文件 ID",
                    "char_start": "起始字元位置（可選）",
                    "char_end":   "結束字元位置（可選）",
                }
            },
            "insert_text": {
                "description": "在 Google Doc 指定位置插入文字",
                "parameters": {
                    "doc_id":   "文件 ID",
                    "text":     "要插入的文字",
                    "position": "插入位置（字元索引）",
                }
            },
            "get_email_thread": {
                "description": "讀取 Gmail 郵件串",
                "parameters": {"thread_id": "郵件串 ID"}
            },
            "update_sheet_range": {
                "description": "更新 Google Sheets 儲存格範圍",
                "parameters": {
                    "sheet_id": "試算表 ID",
                    "range":    "A1 表示法，例如 Sheet1!A1:D10",
                    "values":   "二維陣列",
                }
            },
            "list_document_outline": {
                "description": "取得文件大綱（Heading 結構），不載入全文",
                "parameters": {"doc_id": "文件 ID"}
            },
        }

    # ── 工具實作 ──────────────────────────────────────
    def read_document(self, doc_id: str,
                      char_start: int = 0,
                      char_end: Optional[int] = None) -> dict:
        doc = self._documents.get(doc_id)
        if not doc:
            return {"error": f"文件 {doc_id} 不存在或無讀取權限"}
        sections = [
            s for s in doc["sections"]
            if (char_end is None or s["char_start"] < char_end)
            and s["char_end"] > char_start
        ]
        return {
            "title": doc["title"],
            "sections": sections,
            "total_chars": doc["sections"][-1]["char_end"],
        }

    def list_document_outline(self, doc_id: str) -> dict:
        doc = self._documents.get(doc_id)
        if not doc:
            return {"error": f"文件 {doc_id} 不存在"}
        outline = [
            {"heading": s["heading"],
             "char_start": s["char_start"],
             "char_end":   s["char_end"],
             "preview":    s["content"][:60] + "..."}
            for s in doc["sections"]
        ]
        return {"title": doc["title"], "outline": outline, "cursor_position": doc["cursor_position"]}

    def insert_text(self, doc_id: str, text: str, position: int) -> dict:
        doc = self._documents.get(doc_id)
        if not doc:
            return {"error": f"文件 {doc_id} 不存在"}
        # 模擬插入（生產環境呼叫 batchUpdate API）
        print(f"   [MCP insert_text] 在位置 {position} 插入 {len(text)} 字元")
        return {"status": "ok", "chars_inserted": len(text), "new_position": position + len(text)}

    def get_email_thread(self, thread_id: str) -> dict:
        thread = self._emails.get(thread_id)
        if not thread:
            return {"error": f"郵件串 {thread_id} 不存在"}
        return {"thread_id": thread_id, "messages": thread}

    def call_tool(self, tool_name: str, params: dict) -> dict:
        """MCP 統一入口，dispatch 到各工具"""
        fn = getattr(self, tool_name, None)
        if fn is None:
            return {"error": f"未知工具：{tool_name}"}
        return fn(**params)


# ── 演示：Agent 透過 MCP 操作 Workspace ──────────────
mcp = MockWorkspaceMCPServer()

print("MCP 工具清單（LLM function calling schema）：")
print("=" * 55)
for tool_name, info in mcp.tool_registry.items():
    print(f"  工具：{tool_name}")
    print(f"    說明：{info['description']}")
    print(f"    參數：{list(info['parameters'].keys())}")
    print()

print("\n模擬 Agent 呼叫工具：")
print("-" * 55)

# Agent 先拿大綱
outline = mcp.call_tool("list_document_outline", {"doc_id": "doc_abc123"})
print(f"[1] list_document_outline → 取得 {len(outline['outline'])} 個章節")
for item in outline["outline"]:
    cursor_marker = " ← 游標在這" if outline["cursor_position"] in range(item["char_start"], item["char_end"] + 1) else ""
    print(f"     {item['heading']}{cursor_marker}")

# Agent 讀取目前游標所在章節
section_data = mcp.call_tool("read_document", {"doc_id": "doc_abc123", "char_start": 450, "char_end": 750})
print(f"\n[2] read_document(450~750) → 讀取游標附近章節")
for s in section_data["sections"]:
    print(f"     {s['heading']}: {s['content'][:80]}...")

print("\n✅ MCP 封裝優勢：LLM 呼叫 read_document，不需知道 docs.googleapis.com 的細節")

## 決策 2：Progressive Document Loading vs. 整份文件塞 Context

**❌ 被拒絕的方案：整份文件塞 Context**

每次對話都把完整報告送給 Gemini Pro：
- 顧問的策略報告動輒 50-100 頁，約 80,000–150,000 tokens
- Gemini 1.5 Pro 雖有 1M context，但實際使用中 >128K 準確率下降
- **成本：** 一份 100 頁報告 ≈ 120,000 tokens，$0.00125/1K tokens → 每次對話 $0.15
- 1,000 顧問每人每天 10 次對話 → 每日 $1,500，每月 **$45,000**
- 首 Token 延遲（TTFT）：120K token context 的 prefill 約需 8-12 秒

**✅ 選擇的方案：Progressive Loading（漸進式載入）**

依據用戶**游標位置**和**查詢語意**，動態決定載入哪些段落：
- 永遠載入文件大綱（Heading 結構，~500 tokens）
- 載入游標附近 ±2 個段落（~2,000 tokens）
- 用查詢語意匹配最相關段落（embedding 相似度，~1,500 tokens）
- 總 Context ≈ 4,000–6,000 tokens，成本降低 **95%**

In [ ]:
import math

# ─────────────────────────────────────────────────
# Progressive Document Loader
# ─────────────────────────────────────────────────

@dataclass
class DocumentChunk:
    heading:     str
    content:     str
    char_start:  int
    char_end:    int

    @property
    def token_estimate(self) -> int:
        return len(self.content.split()) * 4 // 3 + 20  # 粗估


class ProgressiveDocumentLoader:
    """
    三層載入策略：
    Layer 1：文件大綱（永遠載入，~500 tokens）
    Layer 2：游標附近段落（±2 段，~2,000 tokens）
    Layer 3：查詢語意相關段落（embedding 匹配，~1,500 tokens）

    目標：把相關 context 控制在 5,000 tokens 以內
    """

    TOKEN_BUDGET = 5000

    def __init__(self, mcp_server: MockWorkspaceMCPServer):
        self.mcp = mcp_server

    def _keyword_similarity(self, query: str, content: str) -> float:
        """模擬 embedding 語意相似度（生產環境用 Vertex AI textembedding-gecko）"""
        q_terms = set(query.lower().replace('？', ' ').replace('，', ' ').split())
        d_terms = set(content.lower().split())
        if not q_terms:
            return 0.0
        return len(q_terms & d_terms) / len(q_terms)

    def load(self, doc_id: str, query: str) -> dict:
        """
        給定查詢和文件 ID，動態決定要載入哪些段落。
        返回：選中的段落 + token 統計
        """
        # ── Layer 1：取得大綱和游標位置 ──────────────
        outline_result = self.mcp.list_document_outline(doc_id)
        cursor_pos = outline_result["cursor_position"]
        all_sections = outline_result["outline"]
        total_chars = all_sections[-1]["char_end"]

        # 找出游標在哪個 section
        cursor_section_idx = 0
        for i, sec in enumerate(all_sections):
            if sec["char_start"] <= cursor_pos < sec["char_end"]:
                cursor_section_idx = i
                break

        # ── Layer 2：游標附近 ±2 段落 ────────────────
        nearby_range = range(
            max(0, cursor_section_idx - 1),
            min(len(all_sections), cursor_section_idx + 2)
        )
        nearby_ids = set(nearby_range)

        # ── Layer 3：語意相關段落 ────────────────────
        full_doc = self.mcp.read_document(doc_id)
        scored_sections = []
        for i, sec in enumerate(full_doc["sections"]):
            score = self._keyword_similarity(query, sec["content"])
            scored_sections.append((i, score, sec))
        scored_sections.sort(key=lambda x: x[1], reverse=True)

        # ── 合併並控制 Token Budget ──────────────────
        selected_indices = set(nearby_ids)
        for idx, score, _ in scored_sections:
            if score > 0.1 and len(selected_indices) < 4:
                selected_indices.add(idx)

        selected_chunks = []
        total_tokens = 300  # 大綱本身的 tokens
        for i in sorted(selected_indices):
            sec = full_doc["sections"][i]
            chunk = DocumentChunk(
                heading=sec["heading"],
                content=sec["content"],
                char_start=sec["char_start"],
                char_end=sec["char_end"],
            )
            if total_tokens + chunk.token_estimate > self.TOKEN_BUDGET:
                # 超出預算：截斷內容
                truncated = sec["content"][:200] + "...[已截斷]"
                chunk = DocumentChunk(sec["heading"], truncated, sec["char_start"], sec["char_end"])
            total_tokens += chunk.token_estimate
            selected_chunks.append(chunk)

        return {
            "outline": [s["heading"] for s in all_sections],
            "cursor_section": all_sections[cursor_section_idx]["heading"],
            "selected_chunks": selected_chunks,
            "total_sections": len(all_sections),
            "loaded_sections": len(selected_chunks),
            "estimated_tokens": total_tokens,
            "full_doc_tokens": total_chars // 4,  # 粗估
        }


# ── 演示 ──────────────────────────────────────────
loader = ProgressiveDocumentLoader(mcp)

queries = [
    "幫我分析第三節的風險點並建議改善措施",
    "執行摘要的重點是什麼？",
    "2026 年 Q1 的執行計劃細節",
]

print("Progressive Loading 演示：")
print("=" * 60)
for q in queries:
    result = loader.load("doc_abc123", q)
    saving = (1 - result["estimated_tokens"] / result["full_doc_tokens"]) * 100
    print(f"\n查詢：'{q}'")
    print(f"  游標位置：{result['cursor_section']}")
    print(f"  載入段落：{result['loaded_sections']}/{result['total_sections']}")
    for chunk in result["selected_chunks"]:
        print(f"    ✓ {chunk.heading} (~{chunk.token_estimate} tokens)")
    print(f"  Token 使用：{result['estimated_tokens']:,} / {result['full_doc_tokens']:,}（節省 {saving:.0f}%）")

print("\n" + "=" * 60)
print("成本對比（1,000 名顧問，每人每日 10 次對話）：")
full_cost  = 1000 * 10 * 30 * (120000 / 1000) * 0.00125
prog_cost  = 1000 * 10 * 30 * (5000  / 1000) * 0.00125
print(f"  整份文件：${full_cost:,.0f}/月")
print(f"  Progressive Loading：${prog_cost:,.0f}/月")
print(f"  月省：${full_cost - prog_cost:,.0f}")

## 決策 3：Vertex AI（資料主權）vs. OpenAI / Claude API

**❌ 被拒絕的方案：OpenAI API 或 Anthropic Claude API**

直接呼叫 `api.openai.com` 或 `api.anthropic.com`：
- 顧問公司的策略報告是 NDA 保護的客戶機密資料
- OpenAI / Anthropic 伺服器在美國，資料跨境傳輸違反客戶合約
- 台灣《個人資料保護法》（PDPA）第 21 條：跨境傳輸個人資料需符合規定
- 稽核要求：無法舉證「我們的資料沒有被 OpenAI 用來訓練」
- 如果客戶是金融機構（銀行、保險），FSC 合規要求更嚴格

**✅ 選擇的方案：Vertex AI（GCP asia-east1，台灣）**

- **資料主權：** Vertex AI 可設定 `location=asia-east1`，確保 inference 和資料儲存都在台灣 GCP Region
- **VPC Service Controls：** 用 VPC-SC perimeter 封鎖資料外傳，即使內部員工也無法繞過
- **CMEK（Customer-Managed Encryption Keys）：** 用 Cloud KMS 管理加密金鑰，Google 也無法讀取
- **稽核日誌：** Cloud Audit Logs 記錄每一次 Vertex AI API 呼叫，符合 SOC 2 要求
- **企業合約：** Google Cloud 可簽 DPA（Data Processing Agreement），有法律約束力

In [ ]:
# ─────────────────────────────────────────────────
# Vertex AI 配置示範
# 生產環境安裝：pip install google-cloud-aiplatform
# ─────────────────────────────────────────────────

VERTEX_CONFIG = {
    "project":  "consulting-ai-prod-tw",  # GCP Project ID
    "location": "asia-east1",             # 台灣 Region，資料不出台灣
    "model":    "gemini-1.5-pro-002",     # 或 gemini-2.0-flash
    "endpoint": "https://asia-east1-aiplatform.googleapis.com",
}

VPC_SC_POLICY = """
# VPC Service Controls 設定（生產環境用）
# gcloud access-context-manager perimeters create consulting-ai-perimeter \
#   --policy=POLICY_ID \
#   --resources=projects/consulting-ai-prod-tw \
#   --restricted-services=aiplatform.googleapis.com,docs.googleapis.com \
#   --access-levels=OFFICE_NETWORK,VPN_ACCESS
#
# 效果：即使有 API key，從 VPC perimeter 外部也無法呼叫 Vertex AI
# 資料完全被封鎖在 GCP asia-east1 內
"""

def simulate_vertex_ai_call(prompt: str, config: dict = VERTEX_CONFIG) -> dict:
    """
    模擬 Vertex AI Gemini Pro 呼叫。
    生產環境：
        import vertexai
        from vertexai.generative_models import GenerativeModel
        vertexai.init(project=config['project'], location=config['location'])
        model = GenerativeModel(config['model'])
        response = model.generate_content(prompt, stream=True)
    """
    return {
        "model":    config["model"],
        "location": config["location"],
        "text":     f"[模擬 Vertex AI 回應] 針對您的問題：{prompt[:50]}...",
        "usage":    {"prompt_tokens": len(prompt.split()) * 4 // 3, "completion_tokens": 150},
    }

# ── 成本對比 ──────────────────────────────────────
print("Vertex AI vs. 第三方 API 決策矩陣：")
print("=" * 70)

comparison = [
    ("資料主權（台灣）",         "✅ asia-east1",                    "❌ 美國伺服器",           "❌ 美國伺服器"),
    ("VPC-SC 支援",             "✅ 原生支援",                       "❌ 不支援",               "❌ 不支援"),
    ("CMEK 加密",               "✅ Cloud KMS",                     "❌ 不支援",               "❌ 不支援"),
    ("稽核日誌",                "✅ Cloud Audit Logs",              "有限支援",               "有限支援"),
    ("GCP IAM 整合",            "✅ 原生",                           "需額外設定",             "需額外設定"),
    ("DPA 合約",                "✅ 可簽",                           "✅ 可簽",                 "✅ 可簽"),
    ("128K context",           "✅ Gemini 1.5 Pro",                "✅ GPT-4o",              "✅ Claude 3.5"),
    ("定價（輸入/1M tokens）",  "$1.25",                            "$2.50",                  "$3.00"),
    ("定價（輸出/1M tokens）",  "$5.00",                            "$10.00",                 "$15.00"),
    ("Google Workspace 整合",   "✅ 原生（同帳單）",                "需要 API 橋接",          "需要 API 橋接"),
]

headers = ["項目", "Vertex AI (Gemini)", "OpenAI GPT-4o", "Anthropic Claude"]
print(f"  {'項目':<25} {'Vertex AI':^22} {'OpenAI GPT-4o':^18} {'Claude':^18}")
print("  " + "-" * 85)
for row in comparison:
    print(f"  {row[0]:<25} {row[1]:^22} {row[2]:^18} {row[3]:^18}")

# 實際成本試算
print("\n每月成本試算（1,000 顧問 × 10 次/日 × 5,000 tokens/次）：")
monthly_tokens_m = 1000 * 10 * 30 * 5000 / 1_000_000  # 百萬 tokens
for name, in_price, out_price in [
    ("Vertex AI Gemini 1.5 Pro", 1.25, 5.00),
    ("OpenAI GPT-4o",            2.50, 10.00),
    ("Anthropic Claude 3.5",     3.00, 15.00),
]:
    cost = monthly_tokens_m * (in_price * 0.7 + out_price * 0.3)
    print(f"  {name:<30} ${cost:>7,.0f}/月")

print(f"\n→ Vertex AI 每月省約 ${(1000*10*30*5000/1_000_000) * ((2.5*0.7+10*0.3) - (1.25*0.7+5*0.3)):,.0f} vs. GPT-4o")

## 決策 4：OAuth 2.0 + Service Account 雙軌認證 vs. 共用 API Key

**❌ 被拒絕的方案：共用 API Key**

用一組 API Key 讓所有顧問呼叫後端：
- **無法稽核：** Key 洩露後不知道是誰在讀取哪份文件
- **權限問題：** 顧問 A 理論上可以讀到顧問 B 的機密報告（API Key 無法區分用戶）
- **資安風險：** 一把 Key 洩露，所有 1,000 顧問的文件存取全部暴露
- **GCP 最佳實踐違反：** Google Cloud Security Best Practices 明確反對共用憑證

**✅ 選擇的方案：OAuth 2.0（用戶身份）+ Service Account（後端操作）**

**雙軌設計：**
1. **OAuth 2.0 Token（代表顧問）：** Google Workspace Add-on 取得用戶的 OAuth token，傳給後端。後端用此 token 呼叫 Docs/Gmail API，確保只能存取「該顧問有權限的文件」。
2. **Service Account（後端內部操作）：** Cloud Run 的 Service Account 負責呼叫 Vertex AI、Pub/Sub 等 GCP 服務，採用 Workload Identity Federation（不需要 Key 檔案）。

**稽核追蹤：** 每次操作都記錄 `user_email` + `doc_id` + `action`，存入 BigQuery。

In [ ]:
# ─────────────────────────────────────────────────
# AuthManager：OAuth 2.0 + Service Account 雙軌
# ─────────────────────────────────────────────────

from datetime import datetime, timedelta

@dataclass
class UserIdentity:
    email:     str
    user_id:   str
    domain:    str
    scopes:    list[str]
    issued_at: datetime = field(default_factory=datetime.utcnow)

    @property
    def is_internal(self) -> bool:
        return self.domain == "consulting-firm.com.tw"

    def can_write_docs(self) -> bool:
        return "https://www.googleapis.com/auth/documents" in self.scopes

    def can_read_gmail(self) -> bool:
        return "https://www.googleapis.com/auth/gmail.readonly" in self.scopes


class AuthManager:
    """
    雙軌認證管理：
    - OAuth 2.0：驗證用戶身份，取得代表用戶的 token
    - Service Account：後端 GCP 資源（Vertex AI, BigQuery）的存取

    生產環境：
    - OAuth token 驗證用 google-auth 的 id_token.verify_oauth2_token()
    - Service Account 用 Workload Identity Federation（不需要 key 檔案）
    """

    # 各角色的 OAuth scope 設定（最小權限原則）
    SCOPE_PROFILES = {
        "consultant": [
            "https://www.googleapis.com/auth/documents",
            "https://www.googleapis.com/auth/gmail.readonly",  # 只讀，不能發送
            "https://www.googleapis.com/auth/spreadsheets",
        ],
        "consultant_readonly": [
            "https://www.googleapis.com/auth/documents.readonly",
            "https://www.googleapis.com/auth/gmail.readonly",
        ],
        "admin": [
            "https://www.googleapis.com/auth/documents",
            "https://www.googleapis.com/auth/gmail.compose",
            "https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive.file",
        ],
    }

    def __init__(self):
        # 模擬 token 快取（生產環境用 Redis / Memorystore）
        self._token_cache: dict[str, UserIdentity] = {}
        self._audit_log: list[dict] = []

    def validate_oauth_token(self, token: str) -> Optional[UserIdentity]:
        """
        驗證 OAuth 2.0 token，返回用戶身份。
        生產環境：呼叫 Google OAuth2 tokeninfo endpoint 或用 id_token.verify_oauth2_token()
        """
        # 模擬 token 驗證
        mock_users = {
            "OAUTH_ALICE_TOKEN": UserIdentity(
                email="alice@consulting-firm.com.tw",
                user_id="uid_alice_001",
                domain="consulting-firm.com.tw",
                scopes=self.SCOPE_PROFILES["consultant"],
            ),
            "OAUTH_BOB_TOKEN": UserIdentity(
                email="bob@consulting-firm.com.tw",
                user_id="uid_bob_002",
                domain="consulting-firm.com.tw",
                scopes=self.SCOPE_PROFILES["consultant_readonly"],
            ),
        }
        return mock_users.get(token)

    def check_permission(self, user: UserIdentity, action: str, resource_id: str) -> bool:
        """
        細粒度權限檢查。
        OAuth token 本身的 scope 是第一層；
        Google Workspace 的 ACL 是第二層（實際存取時 Google 再驗證一次）。
        """
        permission_map = {
            "read_doc":     "https://www.googleapis.com/auth/documents",
            "write_doc":    "https://www.googleapis.com/auth/documents",
            "read_gmail":   "https://www.googleapis.com/auth/gmail.readonly",
            "compose_mail": "https://www.googleapis.com/auth/gmail.compose",
        }
        # 注意：documents.readonly 可以讀但不能寫
        if action == "write_doc":
            return "https://www.googleapis.com/auth/documents" in user.scopes \
                   and "readonly" not in "".join(
                       s for s in user.scopes if "documents" in s
                   )
        required = permission_map.get(action, "")
        return any(required in s for s in user.scopes)

    def audit_log(self, user: UserIdentity, action: str,
                   resource_id: str, result: str) -> None:
        """
        稽核日誌：存入 BigQuery（生產環境）
        schema: user_email, user_id, action, resource_id, result, timestamp
        """
        entry = {
            "user_email":   user.email,
            "user_id":      user.user_id,
            "action":       action,
            "resource_id":  resource_id,
            "result":       result,
            "timestamp":    datetime.utcnow().isoformat() + "Z",
        }
        self._audit_log.append(entry)

    def execute_with_audit(self, token: str, action: str,
                            resource_id: str, fn) -> dict:
        """帶稽核的執行包裝器"""
        user = self.validate_oauth_token(token)
        if not user:
            return {"error": "Invalid OAuth token", "code": 401}

        allowed = self.check_permission(user, action, resource_id)
        result_status = "ALLOWED" if allowed else "DENIED"
        self.audit_log(user, action, resource_id, result_status)

        if not allowed:
            return {"error": f"Permission denied: {user.email} → {action}", "code": 403}

        return {"user": user.email, "result": fn()}


# ── 演示 ──────────────────────────────────────────
auth = AuthManager()

test_cases = [
    ("OAUTH_ALICE_TOKEN", "read_doc",   "doc_abc123", lambda: "已讀取文件"),
    ("OAUTH_ALICE_TOKEN", "write_doc",  "doc_abc123", lambda: "已修改文件"),
    ("OAUTH_BOB_TOKEN",   "read_doc",   "doc_abc123", lambda: "已讀取文件"),
    ("OAUTH_BOB_TOKEN",   "write_doc",  "doc_abc123", lambda: "已修改文件"),  # Bob 無寫入權限
    ("OAUTH_ALICE_TOKEN", "compose_mail","thread_xyz",lambda: "已起草郵件"),  # 超出 scope
    ("INVALID_TOKEN",     "read_doc",   "doc_abc123", lambda: "..."),         # 無效 token
]

print("OAuth 2.0 + Service Account 認證演示：")
print("=" * 65)
for token, action, resource, fn in test_cases:
    result = auth.execute_with_audit(token, action, resource, fn)
    icon = "✅" if "error" not in result else "🚫"
    user_label = token.replace("OAUTH_", "").replace("_TOKEN", "")
    print(f"  {icon} [{user_label}] {action:<15} → {result.get('result', result.get('error', ''))}")

print("\n稽核日誌（將存入 BigQuery）：")
print("-" * 65)
for log in auth._audit_log:
    print(f"  {log['timestamp']} | {log['user_email']:<40} | {log['action']:<15} | {log['result']}")

print("\n✅ 每個操作都有完整稽核軌跡，符合 NDA 稽核要求")

## 決策 5：Streaming Response（SSE）vs. 等全部生成再顯示

**❌ 被拒絕的方案：Wait-for-complete（等全部生成）**

後端等 Gemini Pro 生成完整回答後，一次性回傳給前端：
- **延遲體驗差：** 分析 50 頁報告的風險點需要 10-15 秒，顧問盯著空白畫面等
- **超時風險：** Cloud Run 預設 60 秒 timeout，長回覆可能超時
- **無法取消：** 顧問看到一半覺得不對，無法中途停止，浪費 token
- **記憶體壓力：** 長回答（3,000+ tokens）全存在記憶體後再傳輸

**✅ 選擇的方案：Server-Sent Events（SSE）Streaming**

**為什麼選 SSE 而不是 WebSocket？**
- **SSE 是單向串流（Server → Client），完全符合 AI 生成的場景**
- WebSocket 是雙向，適合即時聊天（多人、互發訊息），AI 回應不需要雙向
- SSE 走標準 HTTP，可透過 CDN / Load Balancer，不需特殊 infrastructure
- Google Workspace Add-on 的 `fetch()` 原生支援 SSE，WebSocket 需要額外函式庫
- SSE 自動重連（`retry:` 欄位），WebSocket 斷線重連需自己實作

| | SSE | WebSocket |
|--|--|--|
| 方向 | 單向（Server→Client） | 雙向 |
| 協議 | HTTP/1.1, HTTP/2 | WS/WSS |
| 自動重連 | ✅ 內建 | 需自實作 |
| CDN 支援 | ✅ | 受限 |
| AI 生成場景 | ✅ 完美符合 | 功能過剩 |
| Google Add-on 支援 | ✅ fetch() 原生 | 需額外引入 |

In [ ]:
# ─────────────────────────────────────────────────
# Streaming Response Handler（SSE 格式）
# ─────────────────────────────────────────────────

import asyncio

class StreamingResponseHandler:
    """
    將 Vertex AI Gemini 的 stream=True 回應轉換為 SSE 格式。

    SSE 格式：
      data: {"token": "台灣"}\n\n
      data: {"token": "電信"}\n\n
      data: {"done": true, "ttft_ms": 320, "total_tokens": 158}\n\n

    生產環境（FastAPI）：
        from fastapi.responses import StreamingResponse
        return StreamingResponse(handler.stream_sse(prompt), media_type="text/event-stream")
    """

    def __init__(self):
        self._cancelled = False

    def cancel(self):
        """用戶按下停止時呼叫，asyncio generator 會在下一個 yield 檢查"""
        self._cancelled = True

    async def _mock_gemini_stream(self, prompt: str) -> AsyncGenerator[str, None]:
        """
        模擬 Vertex AI Gemini stream 回應。
        生產環境：
            model = GenerativeModel("gemini-1.5-pro-002")
            for chunk in model.generate_content(prompt, stream=True):
                yield chunk.text
        """
        response_tokens = [
            "根據", "報告", "第三節", "「競爭風險分析」，", "我識別出", "以下", "四個", "主要風險點：\n\n",
            "**風險一：MVNO 低價競爭**\n",
            "虛擬電信商市佔率從 3% 成長至 8%，", "建議：差異化", "企業專案方案，", "提供 SLA 保證", "讓 MVNO 無法複製。\n\n",
            "**風險二：Google Fi 潛在進入**\n",
            "機率評估為中等（30%），", "建議：強化", "本地化服務綁定", "（繁中支援、", "在地客服）。\n\n",
            "**風險三：LINE Mobile 擴張**\n",
            "建議：評估", "策略合作", "而非對抗，", "LINE 生態系", "有利流量導入。\n\n",
            "**風險四：監管政策收緊**\n",
            "建議：提前", "與 NCC 溝通", "頻譜費用", "調整時程，", "預留財務緩衝。",
        ]
        for token in response_tokens:
            await asyncio.sleep(0.05)  # 模擬 50ms/token 生成速度
            yield token

    async def stream_sse(self, prompt: str) -> AsyncGenerator[str, None]:
        """
        主要 SSE 生成器。回傳的每個字串都是一個 SSE event。
        格式：data: <JSON>\n\n
        """
        self._cancelled = False
        start_time = time.time()
        first_token_time = None
        total_tokens = 0
        full_text = ""

        async for token_text in self._mock_gemini_stream(prompt):
            if self._cancelled:
                yield f'data: {{"cancelled": true}}\n\n'
                return

            if first_token_time is None:
                first_token_time = time.time()
                ttft_ms = int((first_token_time - start_time) * 1000)

            total_tokens += 1
            full_text += token_text
            payload = json.dumps({"token": token_text}, ensure_ascii=False)
            yield f"data: {payload}\n\n"

        # 完成事件
        elapsed_ms = int((time.time() - start_time) * 1000)
        done_payload = json.dumps({
            "done":         True,
            "ttft_ms":      ttft_ms if first_token_time else elapsed_ms,
            "total_ms":     elapsed_ms,
            "total_tokens": total_tokens,
            "tokens_per_s": round(total_tokens / (elapsed_ms / 1000), 1),
        }, ensure_ascii=False)
        yield f"data: {done_payload}\n\n"


# ── 演示：完整 Streaming 流程 ─────────────────────
handler = StreamingResponseHandler()

print("SSE Streaming 演示：")
print("=" * 60)
print("SSE 事件流（前端收到的格式）：")
print("-" * 60)

prompt = "分析報告第三節的競爭風險並建議改善措施"
collected_text = ""
event_count = 0
shown_tokens = 0

async def run_stream():
    global collected_text, event_count, shown_tokens
    async for sse_event in handler.stream_sse(prompt):
        event_count += 1
        data_str = sse_event.replace("data: ", "").strip()
        try:
            data = json.loads(data_str)
        except Exception:
            continue

        if "token" in data:
            collected_text += data["token"]
            if shown_tokens < 5:
                print(f"  data: {{\"token\": \"{data['token']}\"}}")
                shown_tokens += 1
            elif shown_tokens == 5:
                print(f"  data: ... （{event_count} 個事件串流中）")
                shown_tokens += 1

        if data.get("done"):
            print(f"  data: {{\"done\": true, \"ttft_ms\": {data['ttft_ms']}, \"total_tokens\": {data['total_tokens']}, \"tokens_per_s\": {data['tokens_per_s']}}}")
            print("\n" + "=" * 60)
            print("前端最終組裝的完整回應：")
            print("-" * 60)
            print(collected_text)
            print("\n" + "-" * 60)
            print(f"TTFT（Time to First Token）: {data['ttft_ms']} ms")
            print(f"總生成時間: {data['total_ms']} ms")
            print(f"生成速度: {data['tokens_per_s']} tokens/秒")
            print(f"總 SSE 事件數: {event_count}")

await run_stream()

## 完整場景模擬：Google Doc 對話助理工作流程

場景：顧問 Alice 正在編輯《台灣電信業數位轉型策略報告》，游標在第三節。
她透過 Google Docs Add-on 側邊欄輸入問題，AI 助理即時回應並提供插入選項。

完整流程：OAuth 驗證 → Progressive Loading → MCP 工具呼叫 → Vertex AI → SSE 回傳 → 插入文件

In [ ]:
# ─────────────────────────────────────────────────
# 完整 Google Doc 對話助理工作流程
# ─────────────────────────────────────────────────

class GoogleDocChatAssistant:
    """
    整合所有決策的端對端助理。
    生產環境以 Cloud Run 服務部署，FastAPI 提供 REST + SSE endpoints。
    """

    def __init__(self):
        self.auth    = AuthManager()
        self.mcp     = MockWorkspaceMCPServer()
        self.loader  = ProgressiveDocumentLoader(self.mcp)
        self.stream  = StreamingResponseHandler()

    async def chat(self, oauth_token: str, doc_id: str, user_query: str,
                   insert_at_cursor: bool = False) -> dict:
        """
        完整對話流程：
        1. OAuth 驗證
        2. Progressive Loading（取相關段落）
        3. 組裝 Prompt
        4. Vertex AI Streaming
        5. （可選）插回文件
        """
        step_timings = {}
        t0 = time.time()

        # ── Step 1：OAuth 驗證 ────────────────────
        user = self.auth.validate_oauth_token(oauth_token)
        if not user:
            return {"error": "認證失敗", "code": 401}
        print(f"[Step 1] OAuth 驗證完成 → 用戶：{user.email}")
        step_timings["auth_ms"] = int((time.time() - t0) * 1000)

        # ── Step 2：Progressive Loading ──────────
        t1 = time.time()
        context = self.loader.load(doc_id, user_query)
        print(f"[Step 2] Progressive Loading → {context['loaded_sections']}/{context['total_sections']} 段落")
        print(f"         載入段落：{[c.heading for c in context['selected_chunks']]}")
        print(f"         Context tokens：{context['estimated_tokens']:,}（全文 {context['full_doc_tokens']:,}）")
        step_timings["loading_ms"] = int((time.time() - t1) * 1000)

        # ── Step 3：組裝 Prompt ──────────────────
        t2 = time.time()
        doc_context_str = "\n\n".join(
            f"[{chunk.heading}]\n{chunk.content}"
            for chunk in context["selected_chunks"]
        )
        system_prompt = (
            f"你是一位資深管理顧問 AI 助理，協助顧問分析和改善策略報告。\n"
            f"文件標題：{self.mcp._documents[doc_id]['title']}\n"
            f"文件大綱：{' | '.join(context['outline'])}\n"
            f"當前用戶游標位於：{context['cursor_section']}\n\n"
            f"相關文件內容：\n{doc_context_str}\n\n"
            f"請以繁體中文回覆，語氣專業，適合顧問報告風格。"
        )
        full_prompt = f"{system_prompt}\n\n用戶問題：{user_query}"
        print(f"[Step 3] 組裝 Prompt → {len(full_prompt.split()):,} tokens（估算）")
        step_timings["prompt_ms"] = int((time.time() - t2) * 1000)

        # ── Step 4：Vertex AI Streaming ──────────
        t3 = time.time()
        print(f"[Step 4] 呼叫 Vertex AI Gemini Pro（asia-east1）... streaming 中")
        print(f"         資料路徑：GCP asia-east1 內部，不出台灣")
        print()

        # 在模擬環境中用本地 LLM 或固定回應
        ai_response = ""
        if LLM_AVAILABLE:
            msgs = [SystemMessage(content=system_prompt), HumanMessage(content=user_query)]
            resp = await llm.ainvoke(msgs)
            ai_response = resp.content
        else:
            # 使用 streaming handler 的模擬回應
            async for sse_event in self.stream.stream_sse(full_prompt):
                data_str = sse_event.replace("data: ", "").strip()
                try:
                    data = json.loads(data_str)
                    if "token" in data:
                        ai_response += data["token"]
                        print(data["token"], end="", flush=True)
                    elif data.get("done"):
                        pass
                except Exception:
                    pass

        print()
        step_timings["inference_ms"] = int((time.time() - t3) * 1000)

        # ── Step 5：插入文件（可選）──────────────
        insert_result = None
        if insert_at_cursor:
            t4 = time.time()
            cursor_pos = self.mcp._documents[doc_id]["cursor_position"]
            formatted_response = f"\n\n---\n**AI 分析（{datetime.utcnow().strftime('%Y-%m-%d')}）：**\n{ai_response}\n---\n"
            insert_result = self.auth.execute_with_audit(
                oauth_token, "write_doc", doc_id,
                lambda: self.mcp.call_tool("insert_text", {
                    "doc_id":   doc_id,
                    "text":     formatted_response,
                    "position": cursor_pos,
                })
            )
            print(f"[Step 5] 已插入文件（{insert_result.get('result', {}).get('chars_inserted', 0)} 字元）")
            step_timings["insert_ms"] = int((time.time() - t4) * 1000)

        total_ms = int((time.time() - t0) * 1000)

        return {
            "user":           user.email,
            "doc_id":         doc_id,
            "query":          user_query,
            "response":       ai_response[:200] + "..." if len(ai_response) > 200 else ai_response,
            "context_tokens": context["estimated_tokens"],
            "insert_result":  insert_result,
            "timings":        {**step_timings, "total_ms": total_ms},
        }


# ── 執行完整模擬 ──────────────────────────────────
assistant = GoogleDocChatAssistant()

print("完整 Google Doc 對話助理模擬")
print("=" * 65)
print(f"文件：台灣電信業數位轉型策略報告_2025Q4")
print(f"用戶：alice@consulting-firm.com.tw")
print(f"查詢：幫我分析第三節的風險點並建議改善措施")
print("=" * 65)
print()

result = await assistant.chat(
    oauth_token="OAUTH_ALICE_TOKEN",
    doc_id="doc_abc123",
    user_query="幫我分析第三節的風險點並建議改善措施",
    insert_at_cursor=True,
)

print()
print("=" * 65)
print("執行摘要：")
print("-" * 65)
print(f"用戶：{result['user']}")
print(f"Context 使用：{result['context_tokens']:,} tokens")
print(f"各步驟耗時：")
for step, ms in result["timings"].items():
    bar = "█" * (ms // 20)
    print(f"  {step:<18} {ms:>5} ms  {bar}")

## FDE 面試白板語言 — 核心問答

In [ ]:
print("""
FDE 面試：Google Workspace AI 整合系統設計
═══════════════════════════════════════════════════════════

Q: 為什麼用 MCP Server 而不是直接 call Google Workspace API？
A: 直接呼叫的問題有三層：
   (1) 抽象層不對——LLM 不應該知道 HTTP endpoint 格式，
       它應該知道「讀取文件」這個語意，而不是
       GET https://docs.googleapis.com/v1/documents/{id} 這個細節。
   (2) 維護問題——Google Workspace API 版本升級時，
       直接呼叫要改 Prompt，MCP Server 只改 Server 端實作。
   (3) 可測試性——MCP 工具可以 mock，直接呼叫 API 需要真實的 OAuth token。
   MCP 本質是幫 LLM 建立一個乾淨的「工具抽象層」，
   讓 API 的版本和複雜度對 LLM 透明。

───────────────────────────────────────────────────────────

Q: Google Doc 有 50 頁，Context 怎麼管理？
A: 三層漸進式載入策略（Token Budget: 5,000）：
   Layer 1：文件大綱永遠載入（~500 tokens），
             讓 LLM 知道整份文件的結構。
   Layer 2：游標附近 ±1 段落優先載入（~2,000 tokens），
             因為用戶問的問題通常和當前位置相關。
   Layer 3：用查詢語意（textembedding-gecko embedding）
             匹配最相關的其他段落（~1,500 tokens）。
   總計約 4,000-5,000 tokens，比整份文件塞入省 95% 成本，
   且 TTFT 從 8-12 秒降到 1-2 秒。
   關鍵設計：大綱讓 LLM 知道全局，progressive loading 避免超量。

───────────────────────────────────────────────────────────

Q: 顧問公司要求「客戶資料不能出台灣」，你用 GCP 怎麼做到？
A: 四層保護：
   (1) Region 設定：Vertex AI location=asia-east1，
       Cloud Run region=asia-east1，資料不離開台灣 Region。
   (2) VPC Service Controls：建立 perimeter 封鎖
       aiplatform.googleapis.com 和 docs.googleapis.com，
       即使有 API Key 也只能從 VPC 內部呼叫。
   (3) CMEK（Customer-Managed Encryption Keys）：
       Cloud KMS 管理加密金鑰，Google 員工無法讀取資料。
   (4) Cloud Audit Logs：每次 Vertex AI 呼叫都有日誌，
       可向客戶出示稽核報告證明「資料沒有外傳」。
   這套方案可向客戶提交 GCP 合規報告，
   並在合約中附加 Data Processing Agreement。

───────────────────────────────────────────────────────────

Q: Streaming 在技術上怎麼實現？SSE 和 WebSocket 有什麼差別？
A: SSE（Server-Sent Events）格式非常簡單：
     HTTP 200, Content-Type: text/event-stream
     data: {"token": "分析"}\n\n
     data: {"token": "結果"}\n\n
     data: {"done": true, "ttft_ms": 320}\n\n
   
   為什麼 AI 生成場景選 SSE 不選 WebSocket？
   核心原因：AI 生成是單向串流（Server → Client），
   WebSocket 的雙向能力是多餘的功能，還帶來額外複雜度。
   
   具體技術優勢：
   1. SSE 走標準 HTTP，可透過 CDN 和 Cloud Load Balancer，
      不需要設定 WebSocket upgrade 規則。
   2. SSE 內建 retry 機制（data 欄位有 retry: 毫秒數），
      WebSocket 斷線重連需要自己寫。
   3. Google Workspace Add-on 的 fetch() API 原生支援 SSE，
      WebSocket 需要引入額外 polyfill。
   
   什麼時候用 WebSocket？
   多人即時協作（多個用戶互發訊息），
   或需要前端主動推送（取消、進度更新）時才用 WebSocket。
═══════════════════════════════════════════════════════════
""")

## 架構決策速查表

| 決策 | 選擇 | 核心理由 | 量化影響 |
|------|------|----------|----------|
| 整合模式 | MCP Server | LLM 與 API 細節解耦，版本升級只改 Server | 維護成本降低，Prompt 減少 60% |
| Context 載入 | Progressive Loading | 游標位置 + 語意匹配，Token Budget 5K | 成本降低 95%，TTFT 從 10s→2s |
| 模型部署 | Vertex AI asia-east1 | 資料主權，VPC-SC，CMEK | 月省 $40K vs GPT-4o，符合 PDPA |
| 認證架構 | OAuth 2.0 + SA 雙軌 | 用戶身份隔離 + 後端 GCP 最小權限 | 完整稽核日誌，NDA 合規 |
| 回應方式 | SSE Streaming | 單向串流，CDN 友善，原生重連 | TTFT <500ms，用戶體驗大幅提升 |

---

**FDE 面試核心賣點：**
這個設計同時解決了三個層次的問題：
1. **工程問題**（MCP 抽象、Progressive Loading 效能）
2. **合規問題**（Vertex AI 資料主權、OAuth 稽核、VPC-SC）
3. **用戶體驗**（SSE Streaming、即時插入文件）

每個決策都有明確的「為什麼不用 X」—— 這是 Google FDE 面試的核心期待。